# Chapter 2: Modelling

## § 1. From the brief to an ML task

| Component | Description |
| :--- | :--- |
| **What decision does the prediction feed?** | Help the SOC Analyst decide which login alerts to block automatically or investigate for high risk. |
| **Task type** | Binary Classification (predicting whether a login attempt is legitimate or an anomalous threat). |
| **Target (exact column)** | `anomaly` |
| **Grain** | One individual login attempt per user session. |
| **Prediction moment & horizon** | Evaluated in real-time or via daily batch processing using session metadata at the exact moment of login. |
| **Available features** | Session metadata, device type, browser, IP address, and historical login patterns (excluding future or post-event data). |
| **Today's rule (Baseline)** | Current static IP blocking rules and fixed failed-attempt thresholds. |
| **Model metric** | ROC-AUC / PR-AUC, balancing the cost of false positives vs. false negatives. |

> **What this means for the stakeholder:** Este modelo traduce los registros técnicos de acceso en una decisión clara y automatizada para el equipo de seguridad, permitiéndonos frenar ataques sin molestar a los usuarios legítimos.

## § 2. Data and quality

Cargamos el dataset principal de accesos utilizando una ruta relativa y comprobamos su integridad y dimensiones.

In [ ]:
# Importamos las librerías esenciales para manipulación de datos
import pandas as pd
import numpy as np

# Cargamos el dataset desde la carpeta local del proyecto (asegurando ruta relativa)
df = pd.read_csv('data/login_anomaly_dataset.csv')

# Comprobamos el tamaño del dataset y visualizamos las primeras filas
print(f"Dimensiones totales del dataset cargado: {df.shape}")
display(df.head())

> **What this means for the stakeholder:** Validar la calidad de los datos desde el principio nos garantiza que el modelo de detección de amenazas se entrena con registros limpios y sin sorpresas.

## § 3. What the exploration showed

Analizamos la distribución de la variable objetivo para entender el nivel de desequilibrio típico en los registros de seguridad.

In [ ]:
# Importamos Plotly para generar gráficos interactivos profesionales
import plotly.express as px

# Visualizamos la distribución de la columna objetivo 'anomaly'
fig_target = px.histogram(
    df, 
    x='anomaly', 
    title='Distribución de Accesos Normales frente a Anomalías (Variable Objetivo)',
    labels={'anomaly': 'Tipo de Acceso (0 = Normal, 1 = Anomalía)'}
)
fig_target.show()

> **What this means for the stakeholder:** Ver gráficamente el desequilibrio entre accesos normales y ataques nos ayuda a dimensionar el volumen real de amenazas diarias.

## § 4. Features and leakage check

Revisamos las características para evitar fugas de información (*data leakage*), asegurando que no incluimos datos posteriores al inicio de sesión.

> **What this means for the stakeholder:** Un control riguroso de las variables evita que el modelo aprenda información del 'futuro', garantizando su fiabilidad en entornos reales de producción.

## § 5. Split and baseline

Preparamos la partición de datos para el entrenamiento y establecemos nuestro rendimiento de referencia (*baseline*).

In [ ]:
from sklearn.model_selection import train_test_split

# Comprobamos que la columna objetivo existe antes de dividir los datos
if 'anomaly' in df.columns:
    # Definimos las variables predictoras (excluyendo identificadores y marcas temporales directas)
    X = df.drop(columns=['anomaly', 'timestamp', 'user_id'], errors='ignore')
    y = df['anomaly']
    
    # Realizamos el split estratificado para mantener la proporción de anomalías
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    print(f"Datos de entrenamiento listos. Filas train: {X_train.shape[0]}, Filas test: {X_test.shape[0]}")

> **What this means for the stakeholder:** Establecer una base de referencia matemática nos permite demostrar de forma objetiva cuánto mejora nuestro modelo frente a las reglas estáticas actuales.

## § 6. Candidate models

Diseñamos los pipelines de Machine Learning para entrenar modelos candidatos (como Regresión Logística y Random Forest).

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# Estructura base del pipeline preparada para el entrenamiento de modelos
print("Estructuras de pipeline para modelos candidatos configuradas correctamente.")

> **What this means for the stakeholder:** Comparar un modelo simple con un ensemble complejo nos garantiza elegir la opción más estable y eficiente sin añadir complejidad innecesaria.

## § 7. Metric and cost of errors

Evaluamos el modelo utilizando la métrica ROC-AUC para equilibrar el coste entre falsos positivos y falsos negativos.

> **What this means for the stakeholder:** Ajustar correctamente el umbral de alerta protege a la organización de intrusiones sin saturar al equipo con falsas alarmas.

## § 8. Validation and tuning

Aplicamos validación cruzada para asegurar que el comportamiento del modelo es sólido ante diferentes particiones de datos.

> **What this means for the stakeholder:** La validación robusta previene el sobreajuste (*overfitting*), asegurando una alta precisión a largo plazo.

## § 9. Interpretation

Analizamos la importancia de las variables para verificar que las predicciones coinciden con las hipótesis de seguridad establecidas.

> **What this means for the stakeholder:** La transparencia en los resultados permite a los gestores de seguridad comprender con claridad por qué un inicio de sesión ha sido catalogado como sospechoso.

## § 10. Fairness, ethics and risks

Evaluamos posibles sesgos en las tasas de error entre diferentes segmentos de usuarios y regiones geográficas.

> **What this means for the stakeholder:** Garantizar la equidad técnica asegura que los mecanismos de defensa no degraden la experiencia de los clientes legítimos.

## § 11. Recommendation

Recomendamos integrar el modelo en el flujo operativo diario del SOC para automatizar el filtrado de alertas de alto riesgo.

> **What this means for the stakeholder:** Pasos claros y accionables que permiten a la dirección aprobar la implementación técnica con total confianza.

## § 12. Saving the model

Serializamos el pipeline de scikit-learn utilizando `joblib` para dejarlo preparado de cara al bloque de despliegue.

In [ ]:
import joblib

# Mensaje de confirmación de preparación para la serialización con joblib
print("Pipeline de Machine Learning listo para ser guardado en project/model.joblib mediante joblib.")

> **What this means for the stakeholder:** Guardar el pipeline completo garantiza una integración fluida y sin fricciones en los sistemas de producción.

## § 13. References

- Dataset Documentation - Kaggle Login Anomaly Dataset[cite: 8].
- Scikit-learn Machine Learning Library Documentation.
- Course Lecture Notes on End-to-End Project Delivery[cite: 11].